# 71 - Soft Label Training (Face API Confidence as Target Distribution)

**Eksplorasi #5** dari `docs/eksplorasi_lanjutan.md`. Inspired by Liliana et al. (2019) — natural emotions are inherently fuzzy/mixed.

**Ide:** alih-alih `argmax(Face API scores)` → hard one-hot label, pakai **full 7-dim confidence distribution** sebagai target training. Informasi ambiguity yang selama ini dibuang jadi sinyal pembelajaran.

**Setup:**
- Dataset: Primer conf60 (soft labels di `y_{split}_soft.npy` shape (N, 7))
- 4-class remap: `REMAP_4 = [0, 1, 2, 3, 3, 3, 3]` (neutral/happy/sad/negative)
  - Soft labels di-aggregate: `y_soft_4[:, 3] = sum(y_soft_7[:, 3:7])` (sum angry+fearful+disgusted+surprised)
- Arsitektur: **CNN TL** (ResNet18 pretrained ImageNet) — single modality, clean ablation
- Backbone B1 baseline (no class weights, no augmentation) untuk isolate efek soft label saja

**Eksperimen (4-class):**

| Config | Target | Loss | Notes |
|--------|--------|------|-------|
| A (baseline) | Hard (one-hot) | CE | replikasi CNN TL 4c B1 = 0.456 |
| B | Soft (Face API dist) | Soft-CE | soft target, CE-style |
| C | Soft (Face API dist) | KL-divergence | equivalent asymptotically |
| D | Hard + label smoothing ε=0.1 | Smooth CE | baseline smoothing (Szegedy 2016) |

**Output**: `models/frontonly_conf60/soft_label/soft_label_4c_results.json`

**Prerequisites di VPS**:
```bash
python scripts/extract_soft_labels.py
# → data/dataset_frontonly_conf60/y_{train,val,test}_soft.npy
```

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import EmotionCNNTransfer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DATA_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'frontonly_conf60' / 'soft_label'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
PATIENCE = 15
LR_TL = 0.00005

EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']
REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

Device: cuda
GPU: Tesla T4
Setup complete.


In [2]:
# ── Load data ──

def load_split(split):
    img = np.load(DATA_DIR / f'X_{split}_images.npy')
    y = np.load(DATA_DIR / f'y_{split}.npy')
    y_soft = np.load(DATA_DIR / f'y_{split}_soft.npy')
    return img, y, y_soft


def remap_soft_to_4class(y_soft_7):
    """Aggregate soft labels from 7-class to 4-class.
    negative = angry + fearful + disgusted + surprised.
    """
    n = len(y_soft_7)
    y_soft_4 = np.zeros((n, 4), dtype=np.float32)
    y_soft_4[:, 0] = y_soft_7[:, 0]           # neutral
    y_soft_4[:, 1] = y_soft_7[:, 1]           # happy
    y_soft_4[:, 2] = y_soft_7[:, 2]           # sad
    y_soft_4[:, 3] = y_soft_7[:, 3:7].sum(1)  # negative (aggregate)
    return y_soft_4


X_tr, y_tr_7, y_tr_soft7 = load_split('train')
X_v,  y_v_7,  y_v_soft7  = load_split('val')
X_te, y_te_7, y_te_soft7 = load_split('test')

# 4-class labels
y_tr_4 = REMAP_4[y_tr_7]
y_v_4  = REMAP_4[y_v_7]
y_te_4 = REMAP_4[y_te_7]

# 4-class soft labels (aggregate minority → negative)
y_tr_soft4 = remap_soft_to_4class(y_tr_soft7)
y_v_soft4  = remap_soft_to_4class(y_v_soft7)
y_te_soft4 = remap_soft_to_4class(y_te_soft7)

print(f'Train: {X_tr.shape}  hard y dist: {np.bincount(y_tr_4, minlength=4).tolist()}')
print(f'Val:   {X_v.shape}   hard y dist: {np.bincount(y_v_4, minlength=4).tolist()}')
print(f'Test:  {X_te.shape}  hard y dist: {np.bincount(y_te_4, minlength=4).tolist()}')

# Sanity check soft distribution
print(f'\nSoft label stats (train 4c):')
print(f'  sum per sample (should = 1): mean={y_tr_soft4.sum(axis=1).mean():.4f}  min={y_tr_soft4.sum(axis=1).min():.4f}')
print(f'  mean max confidence: {y_tr_soft4.max(axis=1).mean():.4f}')
print(f'  ambiguous (max < 0.7): {(y_tr_soft4.max(axis=1) < 0.7).sum()}/{len(y_tr_4)}')

Train: (5287, 224, 224, 3)  hard y dist: [4526, 416, 287, 58]
Val:   (579, 224, 224, 3)   hard y dist: [477, 52, 24, 26]
Test:  (929, 224, 224, 3)  hard y dist: [688, 183, 50, 8]

Soft label stats (train 4c):
  sum per sample (should = 1): mean=1.0000  min=1.0000
  mean max confidence: 0.9603
  ambiguous (max < 0.7): 166/5287


In [3]:
# ── Dataset yang return (image, hard_label, soft_label) ──

class SoftLabelImageDataset(Dataset):
    def __init__(self, images, y_hard, y_soft):
        # images (N, H, W, 3) float32 → convert ke tensor CHW saat __getitem__
        self.images = images
        self.y_hard = torch.from_numpy(y_hard).long()
        self.y_soft = torch.from_numpy(y_soft).float()

    def __len__(self):
        return len(self.y_hard)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).permute(2, 0, 1).contiguous()
        return img, self.y_hard[idx], self.y_soft[idx]


def make_loader(images, y_hard, y_soft, shuffle=True):
    ds = SoftLabelImageDataset(images, y_hard, y_soft)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=True)


print('Dataset + loader helpers ready.')

Dataset + loader helpers ready.


In [4]:
# ── Loss functions ──

def hard_ce_loss(output, y_hard, y_soft):
    """Standard hard CE — ignore y_soft (baseline)."""
    return F.cross_entropy(output, y_hard)


def soft_ce_loss(output, y_hard, y_soft):
    """Soft cross-entropy with target distribution.
    L = -sum(target_c * log_softmax(output_c))
    """
    log_probs = F.log_softmax(output, dim=1)
    return -(y_soft * log_probs).sum(dim=1).mean()


def kl_div_loss(output, y_hard, y_soft):
    """KL-divergence between model softmax and target soft distribution.
    Equivalent to Soft CE + constant (target entropy).
    """
    log_probs = F.log_softmax(output, dim=1)
    # kl_div expects log-probs as input, probs as target
    return F.kl_div(log_probs, y_soft, reduction='batchmean')


def smooth_ce_loss(output, y_hard, y_soft, eps=0.1, num_classes=4):
    """Label smoothing baseline — smooth hard label uniformly by ε.
    Target: one-hot * (1 - ε) + (ε / K) uniformly.
    """
    with torch.no_grad():
        target = torch.full_like(output, eps / num_classes)
        target.scatter_(1, y_hard.unsqueeze(1), 1.0 - eps + eps / num_classes)
    log_probs = F.log_softmax(output, dim=1)
    return -(target * log_probs).sum(dim=1).mean()


print('Loss functions defined.')

Loss functions defined.


In [5]:
# ── Training loop (custom, supports soft target via loss_fn signature) ──

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for img, y_h, y_s in loader:
        img = img.to(device, non_blocking=True)
        y_h = y_h.to(device, non_blocking=True)
        y_s = y_s.to(device, non_blocking=True)
        out = model(img)
        loss = loss_fn(out, y_h, y_s)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y_h.size(0)
        correct += (out.argmax(1) == y_h).sum().item()
        total += y_h.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn, num_classes):
    model.eval()
    total_loss = 0.0
    all_hard, all_pred = [], []
    for img, y_h, y_s in loader:
        img = img.to(device, non_blocking=True)
        y_h = y_h.to(device, non_blocking=True)
        y_s = y_s.to(device, non_blocking=True)
        out = model(img)
        loss = loss_fn(out, y_h, y_s)
        total_loss += loss.item() * y_h.size(0)
        all_hard.append(y_h.cpu().numpy())
        all_pred.append(out.argmax(1).cpu().numpy())
    y_true = np.concatenate(all_hard)
    y_pred = np.concatenate(all_pred)
    return {
        'loss': total_loss / len(y_true),
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'micro_f1': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'y_true': y_true, 'y_pred': y_pred,
    }


def train_full(model, train_loader, val_loader, test_loader, loss_fn, num_classes,
               save_path, epochs=EPOCHS, patience=PATIENCE, lr=LR_TL):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=8, min_lr=1e-7)

    best_val_f1 = 0.0
    best_epoch = 0
    stale = 0
    history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}

    for epoch in range(1, epochs + 1):
        tl, tacc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val = evaluate(model, val_loader, loss_fn, num_classes)
        scheduler.step(val['macro_f1'])

        history['train_loss'].append(tl)
        history['val_loss'].append(val['loss'])
        history['val_macro_f1'].append(val['macro_f1'])

        improved = val['macro_f1'] > best_val_f1
        if improved:
            best_val_f1 = val['macro_f1']
            best_epoch = epoch
            stale = 0
            torch.save(model.state_dict(), save_path)
        else:
            stale += 1

        print(f'  Epoch {epoch:2d}  train_loss={tl:.4f}  val_loss={val["loss"]:.4f}  '
              f'val_macroF1={val["macro_f1"]:.4f}  {"*" if improved else ""}')

        if stale >= patience:
            print(f'  Early stop at epoch {epoch} (best @ {best_epoch} = {best_val_f1:.4f})')
            break

    # Load best & eval on test
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    test_res = evaluate(model, test_loader, loss_fn, num_classes)
    test_res['best_epoch'] = best_epoch
    test_res['best_val_macro_f1'] = best_val_f1
    test_res['history'] = history
    return test_res


print('Training loop ready.')

Training loop ready.


## Run 4 Configs (4-Class, CNN TL, B1 baseline)

In [6]:
# Build loaders once (reused across all 4 configs)
tr_loader = make_loader(X_tr, y_tr_4, y_tr_soft4, shuffle=True)
v_loader  = make_loader(X_v,  y_v_4,  y_v_soft4,  shuffle=False)
te_loader = make_loader(X_te, y_te_4, y_te_soft4, shuffle=False)

NUM_CLASSES = 4

configs = [
    ('A_hard_CE',       hard_ce_loss),
    ('B_soft_CE',       soft_ce_loss),
    ('C_KL_div',        kl_div_loss),
    ('D_label_smooth',  lambda o, h, s: smooth_ce_loss(o, h, s, eps=0.1, num_classes=NUM_CLASSES)),
]

results = {}
for key, loss_fn in configs:
    print(f"\n{'='*70}")
    print(f'  {key}')
    print(f"{'='*70}")
    model = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
    save_dir = OUTPUT_DIR / f'{NUM_CLASSES}c' / key
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = str(save_dir / 'model.pth')
    res = train_full(model, tr_loader, v_loader, te_loader, loss_fn,
                     NUM_CLASSES, save_path)
    # Simpan tanpa objek numpy besar (y_true/y_pred) di JSON
    results[key] = {
        'accuracy': float(res['accuracy']),
        'macro_f1': float(res['macro_f1']),
        'micro_f1': float(res['micro_f1']),
        'weighted_f1': float(res['weighted_f1']),
        'best_val_macro_f1': float(res['best_val_macro_f1']),
        'best_epoch': int(res['best_epoch']),
    }
    print(f"  → Test: Macro={res['macro_f1']:.4f}  Micro={res['micro_f1']:.4f}  "
          f"Weighted={res['weighted_f1']:.4f}  Acc={res['accuracy']:.4f}")
    print(f"  → Per-class F1:")
    print(classification_report(res['y_true'], res['y_pred'],
                                target_names=EMOTIONS_4, digits=3, zero_division=0))

out_json = OUTPUT_DIR / f'soft_label_{NUM_CLASSES}c_results.json'
with open(out_json, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved all results: {out_json}')


  A_hard_CE


  Epoch  1  train_loss=1.0514  val_loss=0.7381  val_macroF1=0.3451  *


  Epoch  2  train_loss=0.5073  val_loss=0.5997  val_macroF1=0.3327  


  Epoch  3  train_loss=0.3482  val_loss=0.5771  val_macroF1=0.3289  


  Epoch  4  train_loss=0.2652  val_loss=0.6046  val_macroF1=0.2799  


...

  Epoch 33  train_loss=0.3635  val_loss=0.7526  val_macroF1=0.2951  


  Epoch 34  train_loss=0.3630  val_loss=0.7592  val_macroF1=0.3116  


  Epoch 35  train_loss=0.3634  val_loss=0.7530  val_macroF1=0.2884  


  Epoch 36  train_loss=0.3630  val_loss=0.7725  val_macroF1=0.2881  
  Early stop at epoch 36 (best @ 21 = 0.3286)


  → Test: Macro=0.4418  Micro=0.8202  Weighted=0.8089  Acc=0.8202
  → Per-class F1:
              precision    recall  f1-score   support

     neutral      0.873     0.913     0.893       688
       happy      0.713     0.694     0.704       183
         sad      0.219     0.140     0.171        50
    negative      0.000     0.000     0.000         8

    accuracy                          0.820       929
   macro avg      0.451     0.437     0.442       929
weighted avg      0.799     0.820     0.809       929


Saved all results: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/soft_label/soft_label_4c_results.json


## Ringkasan & Comparison

In [7]:
print(f"\n{'='*80}")
print(f'  Soft Label Training — 4-class CNN TL B1 (Primer conf60 test 929 imgs)')
print(f"{'='*80}")
print(f"  {'Config':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Acc':>10} {'Best ep':>8}")
print(f"  {'-'*72}")
for key, r in sorted(results.items(), key=lambda kv: -kv[1]['macro_f1']):
    print(f"  {key:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} "
          f"{r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f} {r['best_epoch']:>8d}")

# Reference baselines
print(f"\nReference (dari eksperimen existing):")
print(f"  CNN TL 4c B1 (hard CE):    Macro F1 = 0.456 (dari models/frontonly_conf60/cnn_tl_4c_B1)")
print(f"  Late Fusion TL 4c B3:       Macro F1 = 0.567 (overall best, dengan augmentation)")
print(f"\nInterpretasi:")
print(f"  - Kalau B/C/D > A: soft target / smoothing membantu di natural data")
print(f"  - Kalau A ≈ B ≈ C: Face API confidence tidak add info beyond hard label")
print(f"  - Kalau D > A tapi B/C < A: label smoothing membantu generic, Face API dist noisy")


  Soft Label Training — 4-class CNN TL B1 (Primer conf60 test 929 imgs)
  Config                      Macro      Micro   Weighted        Acc  Best ep
  ------------------------------------------------------------------------
  C_KL_div                   0.5170     0.8213     0.8258     0.8213        1
  B_soft_CE                  0.4667     0.8407     0.8281     0.8407       14
  D_label_smooth             0.4418     0.8202     0.8089     0.8202       21
  A_hard_CE                  0.4269     0.6986     0.7155     0.6986       23

Reference (dari eksperimen existing):
  CNN TL 4c B1 (hard CE):    Macro F1 = 0.456 (dari models/frontonly_conf60/cnn_tl_4c_B1)
  Late Fusion TL 4c B3:       Macro F1 = 0.567 (overall best, dengan augmentation)

Interpretasi:
  - Kalau B/C/D > A: soft target / smoothing membantu di natural data
  - Kalau A ≈ B ≈ C: Face API confidence tidak add info beyond hard label
  - Kalau D > A tapi B/C < A: label smoothing membantu generic, Face API dist noisy


## Next Steps (kalau hasil promising)

1. **Extend ke Late Fusion TL** — aplikasi soft label ke best model (CNN TL + FCNN fusion), target beat 0.567.
2. **Combine dengan B3 augmentation** — soft CE + weighted + augmented train.
3. **Per-class analysis** — apakah soft label improve F1 kelas minoritas (sad/negative)?
4. **Ablation temperature** — soft label dengan temperature scaling `softmax(Face_API_logits / T)` untuk sharpness control.

Kalau soft label signifikan membantu, jadi kontribusi **novelty unik** untuk tesis (BAB 3 Metodologi + BAB 5 Discussion — ambiguity handling).